[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camenduru/gaussian-splatting-colab/blob/main/gaussian_splatting_colab.ipynb)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_PATH = '/content/drive/MyDrive/warehouse_dataset/warehouse_pics'
print('Files found:', os.listdir(DATASET_PATH))

## Step 2: Install dependencies

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn

!apt-get install -y colmap ffmpeg

## Step 3: Extract frames from any videos, copy images into one folder

In [ ]:
import os, shutil, glob

IMAGES_DIR = '/content/my_scene/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

files = os.listdir(DATASET_PATH)
frame_counter = 0

for f in files:
    src = os.path.join(DATASET_PATH, f)
    ext = os.path.splitext(f)[1].lower()

    if ext in VIDEO_EXTS:
        print(f'Extracting frames from video: {f}')
        out_pattern = os.path.join(IMAGES_DIR, f'vid_{frame_counter:04d}_%04d.png')
        # Extract 2 frames per second — increase fps= for more detail
        os.system(f'ffmpeg -i "{src}" -vf fps=2 "{out_pattern}" -hide_banner -loglevel error')
        frame_counter += 1

    elif ext in IMAGE_EXTS:
        shutil.copy(src, IMAGES_DIR)

total = len(os.listdir(IMAGES_DIR))
print(f'Total images ready for COLMAP: {total}')

## Step 4: Run COLMAP to compute camera poses

In [ ]:
SCENE_DIR = '/content/my_scene'
DB_PATH   = f'{SCENE_DIR}/colmap.db'

os.makedirs(f'{SCENE_DIR}/sparse', exist_ok=True)

# Feature extraction
!colmap feature_extractor \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --ImageReader.single_camera 1

# Feature matching
!colmap exhaustive_matcher \
    --database_path {DB_PATH}

# Sparse reconstruction
!colmap mapper \
    --database_path {DB_PATH} \
    --image_path {IMAGES_DIR} \
    --output_path {SCENE_DIR}/sparse

print('COLMAP done. Sparse models:', os.listdir(f'{SCENE_DIR}/sparse'))

## Step 5: Train Gaussian Splatting

In [ ]:
%cd /content/gaussian-splatting

# COLMAP mapper outputs to sparse/0, sparse/1, etc. — use the first model
!python train.py -s /content/my_scene --model_path /content/my_scene/output

## Step 6 (Optional): Render and export video

In [ ]:
!python render.py -m /content/my_scene/output

!ffmpeg -framerate 24 \
    -i /content/my_scene/output/train/ours_30000/renders/%05d.png \
    -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" \
    -c:v libx264 -r 24 -pix_fmt yuv420p \
    /content/warehouse_render.mp4 -y

print('Render saved to /content/warehouse_render.mp4')

## Step 7 (Optional): Copy output back to Google Drive

In [ ]:
import shutil
OUTPUT_DRIVE = '/content/drive/MyDrive/warehouse_dataset/warehouse_output'
shutil.copytree('/content/my_scene/output', OUTPUT_DRIVE, dirs_exist_ok=True)
print('Output copied to Google Drive:', OUTPUT_DRIVE)